# Advanced 07 — HyDE: Imagine a Document Before Searching

**Outcome:** decide when a hypothetical document earns its extra work without ever treating generated text as evidence.

This lab deliberately separates three layers:

1. a hand-authored, gold-aware **mechanism fixture**;
2. a credential-free **dense retrieval experiment** over 36 labelled cases and 12 slices; and
3. an optional **local text-to-text generator experiment** for model-behaviour inspection.

The proof population and the measurement population are not interchangeable. Tenant, classification, and lifecycle authorization define the candidate universe before any retrieval leg runs.

## 1. Load the reusable implementation

The notebook imports the same `lab.py` used by tests. It works when launched either from this course folder or from the repository root.

In [ ]:
import importlib.util
import os
from dataclasses import asdict
from pathlib import Path
import sys

module_candidates = [Path("lab.py"), Path("curriculum/advanced/07-hyde-retrieval/lab.py")]
module_path = next(path for path in module_candidates if path.exists())
spec = importlib.util.spec_from_file_location("hyde_lab", module_path)
hyde = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = hyde
spec.loader.exec_module(hyde)

len(hyde.CORPUS), len(hyde.MECHANISM_CASES), len(hyde.EVALUATION_CASES)

In [ ]:
slices = sorted({case.query_type for case in hyde.EVALUATION_CASES})
slice_counts = {
    query_type: sum(case.query_type == query_type for case in hyde.EVALUATION_CASES)
    for query_type in slices
}
slice_counts

## 2. Layer 1 — prove the mechanism with transparent TF-IDF

These six cases intentionally use hand-authored hypotheses that know the target vocabulary. They show *how* representation change, fusion, and drift work. They must not be reported as evidence that a language model improves retrieval.

In [ ]:
mechanism_index = hyde.build_index(backend="tfidf")
mechanism_rows = {}
for strategy in ("original", "hyde", "hyde_plus_original"):
    rows = hyde.evaluate(
        strategy,
        cases=hyde.MECHANISM_CASES,
        top_k=1,
        generator=hyde.generate_mechanism_hypotheses,
        index=mechanism_index,
        rerank=False,
    )
    mechanism_rows[strategy] = rows
    print(strategy, hyde.summarize(rows))

In [ ]:
query = "Why does the app log me out overnight?"
trace = hyde.retrieve(
    query,
    strategy="hyde_plus_original",
    hypothesis_count=3,
    generator=hyde.generate_mechanism_hypotheses,
    index=mechanism_index,
    top_k=3,
    rerank=True,
)
{
    "hypotheses_are_search_only": trace.hypotheses,
    "real_evidence_ids": trace.evidence_ids,
    "generator_call_proxy": trace.generator_call_proxy,
    "retrieval_legs": trace.retrieval_legs,
    "fusion_operations": trace.fusion_operations,
    "reranked_candidates": trace.reranked_candidates,
}

## 3. Layer 2 — repeat the experiment with real local dense embeddings

`SentenceTransformerEncoder` uses `encode_query` for the original query and `encode_document` for corpus passages and hypothetical passages. Vectors are normalized before cosine-equivalent dot-product search. The model downloads locally and needs no API key.

The deterministic generic generator below is deliberately **not gold-aware**. It is a controlled representation ablation; the optional local-generator section later is the model-behaviour experiment.

In [ ]:
dense_index = hyde.build_index(backend="dense")
print(dense_index.backend_name, len(dense_index.documents))

In [ ]:
dense_rows = {}
for strategy in ("original", "hyde", "hyde_plus_original", "conditional"):
    rows = hyde.evaluate(
        strategy,
        cases=hyde.EVALUATION_CASES,
        top_k=3,
        hypothesis_count=3,
        generator=hyde.generate_generic_hypotheses,
        index=dense_index,
        rerank=True,
    )
    dense_rows[strategy] = rows
    print(strategy, hyde.summarize(rows))

### Read aggregate and slice metrics together

`Recall@k` and reciprocal rank are undefined for an empty relevance set. The summary excludes those rows from retrieval averages and reports abstention behaviour separately as `no_answer_accuracy`. This prevents a no-answer case from receiving a fabricated perfect recall.

In [ ]:
conditional_slices = hyde.summarize_by_slice(dense_rows["conditional"])
for query_type, metrics in conditional_slices.items():
    print(query_type, metrics)

In [ ]:
no_answer_rows = [
    row for row in dense_rows["conditional"]
    if row.query_type == "no_answer"
]
[(row.case_id, row.recall_at_k, row.no_answer_correct, row.retrieved_ids) for row in no_answer_rows]

## 4. Evaluate the router independently

Retrieval quality cannot reveal whether a conditional router made the right decision. The router has its own labels and scorecard, including false-positive/false-negative rates and a hard safety metric for exact IDs, policy IDs, numbers, dates, and proprietary acronyms.

In [ ]:
router_rows = hyde.evaluate_router()
router_summary = hyde.summarize_router(router_rows)
router_summary

In [ ]:
[asdict(row) for row in router_rows if row.expected_strategy != row.predicted_strategy]

## 5. Authorization is a pre-retrieval boundary

The corpus contains three tempting matches that belong to another tenant, exceed the caller's classification ceiling, or are retired. They are filtered before the index is built—not retrieved and removed afterward.

In [ ]:
forbidden_ids = {"tenant-b-auth-match", "restricted-auth-match", "retired-auth-match"}
authorized = hyde.authorized_documents(hyde.CORPUS, hyde.DEFAULT_ACCESS)
auth_trace = hyde.retrieve(
    "Why does the app log me out overnight?",
    strategy="hyde_plus_original",
    hypothesis_count=3,
    generator=hyde.generate_generic_hypotheses,
    index=dense_index,
)
{
    "authorized_document_count": len(authorized),
    "forbidden_indexed": sorted(forbidden_ids & {doc.doc_id for doc in dense_index.documents}),
    "forbidden_returned": sorted(forbidden_ids & set(auth_trace.evidence_ids)),
    "unauthorized_candidate_count": auth_trace.unauthorized_candidate_count,
}

## 6. Turn drift into a baseline regression

The mechanism fixture intentionally imagines `ZX-47` as a drug even though the corpus defines it as a hardware controller. Compare the candidate strategy with the original-query baseline and fail the gate when a previously successful answerable case becomes a miss.

In [ ]:
zx_case = [case for case in hyde.MECHANISM_CASES if case.case_id == "mechanism-zx47"]
zx_baseline = hyde.evaluate("original", cases=zx_case, top_k=1, index=mechanism_index, rerank=False)
zx_candidate = hyde.evaluate(
    "hyde",
    cases=zx_case,
    top_k=1,
    generator=hyde.generate_mechanism_hypotheses,
    index=mechanism_index,
    rerank=False,
)
{
    "baseline": zx_baseline[0].retrieved_ids,
    "candidate": zx_candidate[0].retrieved_ids,
    "drift": hyde.drift_metrics(zx_baseline, zx_candidate),
}

## 7. Diversity and work are measurable, not decorative

Multiple hypotheses are useful only when they add perspectives. Compare uniqueness, lexical overlap, embedding similarity, and top-k retrieval overlap. Then account for generation calls, retrieval legs, fusion operations, and reranked candidates separately; this is a work proxy, not a currency estimate.

In [ ]:
diverse_hypotheses = hyde.generate_generic_hypotheses(
    "Can I work overseas and what reviews do I need?",
    count=3,
)
asdict(hyde.analyze_hypothesis_diversity(diverse_hypotheses, index=dense_index, top_k=3))

In [ ]:
work_scorecard = {
    strategy: {
        key: summary[key]
        for key in ("generation_call_proxy", "retrieval_legs", "fusion_operations", "reranked_candidates")
    }
    for strategy, rows in dense_rows.items()
    for summary in [hyde.summarize(rows)]
}
work_scorecard

## 8. Layer 3 — optional real local-generator experiment

Set `RUN_HYDE_LOCAL_GENERATOR=1` before launching Jupyter to download `google/flan-t5-small`, generate hypotheses locally, and evaluate a balanced six-case sample. This section is separate from the mechanism fixture: it reports what the model actually emits, including weak or repetitive passages. It uses no API key, but model download and CPU generation add runtime.

In [ ]:
RUN_LOCAL_GENERATOR = os.getenv("RUN_HYDE_LOCAL_GENERATOR", "0") == "1"
local_generator_result = {"status": "skipped", "enable_with": "RUN_HYDE_LOCAL_GENERATOR=1"}

if RUN_LOCAL_GENERATOR:
    local_generator = hyde.LocalText2TextHypothesisGenerator()
    model_cases = tuple(
        case for case in hyde.EVALUATION_CASES
        if case.case_id in {"sg-01", "pp-01", "wf-01", "ei-01", "num-01", "mp-01"}
    )
    local_rows = hyde.evaluate(
        "hyde_plus_original",
        cases=model_cases,
        top_k=3,
        hypothesis_count=3,
        generator=local_generator,
        index=dense_index,
        rerank=True,
    )
    sample_hypotheses = local_generator(model_cases[0].query, count=3)
    local_generator_result = {
        "status": "executed",
        "model": local_generator.model_name,
        "metrics": hyde.summarize(local_rows),
        "sample_hypotheses": sample_hypotheses,
        "diversity": asdict(hyde.analyze_hypothesis_diversity(sample_hypotheses, index=dense_index)),
    }

local_generator_result

## 9. Release gates

A production candidate should be rejected if it leaks unauthorized candidates, routes high-risk queries through HyDE, regresses successful baseline cases beyond the agreed threshold, or spends extra generation/retrieval work without a slice-level gain.

In [ ]:
assert len(hyde.EVALUATION_CASES) == 36
assert len(slices) == 12
assert all(row.recall_at_k is None for row in no_answer_rows)
assert router_summary["high_risk_query_to_hyde_rate"] == 0.0
assert auth_trace.unauthorized_candidate_count == 0
assert not (forbidden_ids & set(auth_trace.evidence_ids))
assert hyde.drift_metrics(zx_baseline, zx_candidate)["baseline_regression_count"] == 1

print("HyDE course release gates passed.")

## Continue learning

- Read the course README for architecture patterns, failure modes, implementation guidance, and research context.
- Replace the local fixture with your application-owned authorization predicate and evaluation set.
- Continue to [Advanced 08 — Enterprise RAG Platform Capstone](../08-enterprise-rag-capstone/README.md) to integrate retrieval, governance, observability, and release controls.